# SI7016 — Lecture 06 - RN Labs (GPU T4)
## Embeddings + BERT + GPT/LLMs abiertos con Hugging Face y LangChain

**Tema:** Redes neuronales para NLP — del encoder (BERT) al decoder (GPT/LLMs)  


> Este notebook está diseñado para ejecutarse en Google Colab con **NVIDIA T4 (16GB)**.

### Actividades

1. Construir y comparar representaciones (TF‑IDF vs embeddings neuronales).
2. Entender y usar embeddings contextuales con BERT (token embeddings) y sus aplicaciones.
3. Ejecutar un LLM abierto estilo GPT (decoder‑only) para generación y tareas
4. Ejemoplo básico de un RAG (retrieve → prompt → generate) usando embeddings + LangChain. Más adelante en AWS Bedrock + opensearch
5. mini‑reto: comparar estrategias de retrieval y justificar decisiones.

In [ ]:
# Instalar dependencias (puede que ya las tenga tu ambiente, tipo google colab)

!pip -q install -U "transformers>=4.41" "datasets>=2.19" "accelerate>=0.30" "sentence-transformers>=3.0"     "scikit-learn>=1.3" "pandas>=2.0" "matplotlib>=3.7"     "langchain>=0.2" "langchain-community>=0.2" "langchain-core>=0.2"

# (Opcional) Cuantización 4-bit para modelos más pesados.
!pip -q install -U bitsandbytes

In [ ]:
# verificar disponibilidad de GPU
import os, re, math, json, textwrap
import numpy as np
import pandas as pd

import torch
from sklearn.metrics.pairwise import cosine_similarity

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

---
## Responda
## ¿Qué es un embedding y para qué sirve?

comparar:
- TF‑IDF (clásico basado en vocabulario)
- Sentence embeddings (neuronales, semánticos)


In [ ]:
# ejemplo con este dataset básico, pero en el reto podrá utilizar un dataset más completo como ag_news
texts = [
    "El café liofilizado preserva aroma y sabor.",
    "La liofilización reduce la humedad del café.",
    "Me gusta el espresso por su sabor intenso.",
    "La Universidad EAFIT está en Medellín.",
    "BERT es un modelo encoder para comprensión de texto.",
    "GPT es un modelo decoder para generación de texto.",
    "El secado por congelación conserva compuestos volátiles.",
    "Los transformers usan atención para modelar contexto."
]

pairs_to_compare = [
    (0, 1),  # café liofilizado vs liofilización
    (0, 6),  # café liofilizado vs secado por congelación (sinónimo aproximado)
    (4, 5),  # BERT vs GPT (relacionados pero distintos)
    (0, 3),  # café vs EAFIT (no relacionados)
]

len(texts), texts[:2]

In [ ]:
# representaciones en tfidf y realización de función de similitud coseno
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(texts)
cos_tfidf = cosine_similarity(X)

def show_pairs(sim_matrix, name):
    print(f"\n== {name} ==")
    for i,j in pairs_to_compare:
        print(f"({i},{j}) {sim_matrix[i,j]:.3f} | '{texts[i]}'  <->  '{texts[j]}'")

show_pairs(cos_tfidf, "TF-IDF cosine")

# RESPONDA: CONCLUSIONES DE ACUERDO A LOS RESULTADOS

In [ ]:
# utilizar un modelo de embeddings para representrar las oraciones del dataset texts
from sentence_transformers import SentenceTransformer

# Modelo multi-lingüe liviano y muy efectivo para similitud semántica
st_model_id = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st = SentenceTransformer(st_model_id)

E = st.encode(texts, normalize_embeddings=True)
cos_st = E @ E.T

show_pairs(cos_st, "Sentence-Transformers cosine")
# RESPONDA: CONCLUSIONES DE ACUERDO A LOS RESULTADOS

# Busque en la libreria de modelos de huggingface otros modelos de embeddgins y compare resultados

### Responda de forma corta (3–5 líneas)
1) ¿Qué pares mejoraron más al pasar de TF‑IDF a embeddings?  
2) ¿Qué limita a TF‑IDF en semántica?  
3) ¿Qué “tipo de embedding” conviene para Recuperación semántico?

---

## 2) BERT como encoder: embeddings contextuales y aplicaciones

Comprobar:

- Como cambia el embedding de una misma palabra según el contexto.
- Uso práctico de BERT en las tareas de NER y clasificación con Hugging Face pipelines.

In [ ]:
from transformers import AutoTokenizer, AutoModel

# modelo BERT más basico
bert_id = "bert-base-multilingual-cased"
tok = AutoTokenizer.from_pretrained(bert_id)
bert = AutoModel.from_pretrained(bert_id)

bert.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
bert.to(device)

bert_id, device

### Embedding de una palabra depende del contexto

Ejemplo: la palabra “banco” (financiero vs asiento).
Vamos a extraer el embedding del token (promediando sub‑tokens si aplica) y comparar.

In [ ]:
def token_embedding(sentence, target):
    inputs = tok(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        out = bert(**inputs).last_hidden_state[0]  # (seq_len, hidden)
    tokens = tok.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

    # Encuentra posiciones de sub-tokens que contienen la palabra objetivo
    idx = [i for i,t in enumerate(tokens) if target in t.lower()]
    if not idx:
        raise ValueError(f"No encontré '{target}' en tokens: {tokens}")

    emb = out[idx].mean(dim=0).detach().cpu().numpy()
    emb = emb / np.linalg.norm(emb)
    return emb, tokens

s1 = "El banco central anunció una nueva medida económica."
s2 = "Me senté en el banco del parque para descansar."
#s2 = "Me senté en el banco del parque para descansar de camino al banco."

e1, t1 = token_embedding(s1, "banco")
e2, t2 = token_embedding(s2, "banco")

cos = float(e1 @ e2)
print("Cosine(banco_financiero, banco_asiento) =", round(cos, 3))
print("Tokens s1:", t1)
print("Tokens s2:", t2)

Responda:  
¿Por qué BERT puede producir embeddings distintos para la misma palabra en diferentes oraciones?

### NER con Hugging Face pipeline (modelo ya entrenado)

usar un modelo multilingüe para NER y ver qué entidades detecta.

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="Davlan/bert-base-multilingual-cased-ner-hrl",
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

text_ner = "La Universidad EAFIT está en Medellín. Google lanzó BERT en 2018 y OpenAI lanzó GPT-3 en 2020."
entities = ner(text_ner)
pd.DataFrame(entities)

Análisis:  
1) ¿Qué entidades detectó bien?  
2) ¿Qué errores observas y cómo los explicarías?  
3) ¿Qué harías para mejorar NER en un dominio específico (ej. medicina, legal, ingeniería)?
4) usa otras oraciones para verificar aciertos y errores

### Clasificación (análisis de sentimientos) con pipeline

Usaremos un modelo multilingüe pre‑entrenado para sentimiento.

In [ ]:
clf = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=0 if torch.cuda.is_available() else -1
)

samples = [
    "Este curso es excelente, la explicación de transformers fue muy clara.",
    "No me gustó el laboratorio, estuvo confuso y demasiado largo.",
    "La liofilización del café conserva el aroma de manera sorprendente."
]
pd.DataFrame([{"text": s, **clf(s)[0]} for s in samples])

Responde:   
¿Por qué BERT suele ser más fuerte en comprensión (clasificación/NER) que en generación?

---

## 3) LLMs abiertos (Hugging Face)

- generación autoregresiva (continuación de texto)
- prompts “cero‑shot” vs “few‑shot”
- control de formato (bullets / JSON)

> Modelo sugerido para T4: Qwen2.5 1.5B Instruct (ligero) o Phi‑3 mini.  
> cuantización 4‑bit para evitar OOM.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Modelo abierto liviano: para nvidia T4.
# Otro modelo: "microsoft/phi-3-mini-4k-instruct"
llm_id = "Qwen/Qwen2.5-1.5B-Instruct"

tok_llm = AutoTokenizer.from_pretrained(llm_id, use_fast=True)

if torch.cuda.is_available():
    llm = AutoModelForCausalLM.from_pretrained(
        llm_id,
        device_map="auto",
        load_in_4bit=True,
        torch_dtype=torch.float16
    )
else:
    llm = AutoModelForCausalLM.from_pretrained(llm_id)

gen = pipeline(
    "text-generation", # tarea
    model=llm,
    tokenizer=tok_llm,
    device_map="auto" if torch.cuda.is_available() else None
)

llm_id

In [ ]:
# Prompt 1: cero-shot
prompt_0 = "Resume en 5 puntos: El café liofilizado preserva aroma y reduce humedad, mejorando estabilidad y vida útil."

out0 = gen(prompt_0, max_new_tokens=120, do_sample=True, temperature=0.7)
print(out0[0]["generated_text"])

In [ ]:
# Prompt 2: few-shot (ejemplo de formato)
prompt_fs = """Tarea: Resume en 5 puntos.

Ejemplo:
Texto: La acuarela urbana combina dibujo y pintura rápida en exteriores.
Resumen:
- Técnica rápida en exteriores
- Combina dibujo y color
- Enfatiza observación
- Requiere síntesis visual
- Útil para registrar escenas

Ahora:
Texto: El café liofilizado preserva aroma y reduce humedad, mejorando estabilidad y vida útil.
Resumen:
"""

out1 = gen(prompt_fs, max_new_tokens=150, do_sample=True, temperature=0.6)
print(out1[0]["generated_text"])

Responde:  
¿Qué cambió al usar few‑shot? ¿El modelo siguió mejor el formato? ¿Por qué?

---

## 4) Mini‑RAG local: Embeddings + Retrieval + (LangChain Prompt)

RAG mínimo con:
- Chunks pequeños (corpus local)
- Embeddings de sentence-transformers
- Recuperación top‑k por coseno
- Prompt con LangChain (solo para estructura)

> Esto NO usa OpenSearch ni FAISS: es el “mínimo viable” para comprender el flujo.

In [ ]:
# Mini-corpus (puedes reemplazarlo por tus propios textos o de de datasets de texto de huggingface)
chunks = [
    "La liofilización es un proceso de secado por congelación que elimina agua por sublimación.",
    "En café liofilizado, la reducción de humedad mejora la estabilidad y vida útil del producto.",
    "El proceso busca conservar aroma y compuestos volátiles del café mediante bajas temperaturas.",
    "BERT es un encoder basado en transformers, útil para clasificación, NER y extracción de información.",
    "GPT y LLMs decoder-only son fuertes en generación y seguimiento de instrucciones.",
    "Los embeddings semánticos permiten búsqueda por significado, no solo por coincidencia de palabras.",
    "Transformers usan self-attention para capturar dependencias largas y paralelizar entrenamiento.",
    "En RAG, recuperamos fragmentos relevantes y luego generamos respuesta con un LLM usando ese contexto."
]

E_chunks = st.encode(chunks, normalize_embeddings=True)

def retrieve(query, k=3):
    q = st.encode([query], normalize_embeddings=True)
    scores = (E_chunks @ q.T).ravel()
    idx = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in idx]

question = "¿Qué ventajas tiene la liofilización en el café?"
print(question)
retrieved = retrieve(question, k=3)
print(pd.DataFrame(retrieved, columns=["chunk", "score"]))

print(question)
question = "¿Qué desventajas tiene la liofilización en el café?"
retrieved = retrieve(question, k=3)
print(pd.DataFrame(retrieved, columns=["chunk", "score"]))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template(
"""Eres un asistente académico. Responde SOLO usando el contexto.
Si la respuesta no está en el contexto, di: "No está en el contexto proporcionado."

Contexto:
{context}

Pregunta: {question}

Respuesta en español (máx 6 líneas):
""")

context = "\n\n".join([c for c,_ in retrieved])
final_prompt = rag_prompt.format(context=context, question=question)
print(final_prompt)

In [ ]:
# Generamos respuesta con el LLM abierto usando el prompt construido
out = gen(final_prompt, max_new_tokens=180, do_sample=False, temperature=0.0)
print(out[0]["generated_text"])

Responde:  
1) ¿Qué ocurre si el top‑k no incluye la información?  
2) ¿Cómo reducirías alucinaciones en un RAG real?  
3) ¿Qué componente es más importante: embeddings, retrieval, prompt o el LLM?

---

# 5) Mini‑Reto Clase
## “Embeddings para Retrieval: Sentence‑Transformers vs BERT (CLS/mean)”

**Objetivo:** comparar dos estrategias de embeddings para recuperación semántica.

### Estrategias
- A) Sentence‑Transformers (modelo ya usado)
- B) BERT embeddings artesanales o  (promedio de tokens / CLS)

### Tareas
1) Define un conjunto de preguntas sobre el corpus (8–10 preguntas).  
2) Para cada pregunta, recupera top‑3 chunks con A y con B.  
3) Define una “verdad base” simple: para cada pregunta indica el índice del chunk correcto (puede ser manual).  
4) Calcula Recall@3 para A y B.  
5) Escribe un análisis breve:
   - 2 casos donde A gana y por qué  
   - 2 casos donde B gana (si ocurre) y por qué  
   - conclusión: ¿qué usarías en una app real y por qué?
6) si tienes tiempo, usa un dataset grande de huggingface, ojala en español para QA y repite este ejercicio.

In [ ]:
# Define preguntas y ground-truth (chunk correcto)
questions = [
    "¿Qué es la liofilización?",
    "¿Para qué sirve reducir la humedad en café liofilizado?",
    "¿Por qué se conservan compuestos volátiles?",
    "¿Qué es RAG en pocas palabras?",
    "¿Para qué sirve BERT?",
    "¿Para qué sirve GPT en comparación con BERT?",
    "¿Qué aportan los embeddings semánticos en búsqueda?",
    "¿Qué hace especial a self-attention en transformers?"
]

# Índice del chunk correcto (0..len(chunks)-1)
# (Puedes modificar si ajustas el corpus.)
gold = [0, 1, 2, 7, 3, 4, 5, 6]

assert len(questions) == len(gold)
len(questions)

In [ ]:
# Estrategia A: Sentence-Transformers
E_A = st.encode(chunks, normalize_embeddings=True)

def retrieve_A(query, k=3):
    q = st.encode([query], normalize_embeddings=True)
    scores = (E_A @ q.T).ravel()
    idx = np.argsort(scores)[::-1][:k]
    return idx.tolist(), scores[idx].tolist()

# 5C) Estrategia B: BERT mean pooling (simple)
def bert_sentence_embedding(sentence):
    inputs = tok(sentence, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = bert(**inputs).last_hidden_state  # (1, seq, hidden)
    # mean pooling sobre tokens (excluyendo padding si existiera)
    mask = inputs["attention_mask"].unsqueeze(-1)  # (1, seq, 1)
    emb = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    emb = emb[0].detach().cpu().numpy()
    emb = emb / np.linalg.norm(emb)
    return emb

E_B = np.stack([bert_sentence_embedding(c) for c in chunks], axis=0)

def retrieve_B(query, k=3):
    q = bert_sentence_embedding(query)
    scores = E_B @ q
    idx = np.argsort(scores)[::-1][:k]
    return idx.tolist(), scores[idx].tolist()

# Prueba rápida
retrieve_A("¿Qué es RAG?", k=3), retrieve_B("¿Qué es RAG?", k=3)[:1]

In [ ]:
def recall_at_k(retrieved_idx, gold_idx, k=3):
    return 1.0 if gold_idx in retrieved_idx[:k] else 0.0

rows = []
for q, g in zip(questions, gold):
    idxA, _ = retrieve_A(q, k=3)
    idxB, _ = retrieve_B(q, k=3)
    rows.append({
        "question": q,
        "gold_chunk": g,
        "A_top3": idxA,
        "B_top3": idxB,
        "A_hit": recall_at_k(idxA, g, 3),
        "B_hit": recall_at_k(idxB, g, 3)
    })

df = pd.DataFrame(rows)
df

In [ ]:
A_recall = df["A_hit"].mean()
B_recall = df["B_hit"].mean()
print("Recall@3 (A: Sentence-Transformers):", round(A_recall, 3))
print("Recall@3 (B: BERT mean pooling):", round(B_recall, 3))

### Responde:

- ¿A o B para retrieval semántico y por qué?
- 2 casos donde A gana (menciona pregunta + explicación)
- 2 casos donde B gana (si aplica)
- ¿Qué cambiarías para mejorar BERT como embedding de retrieval? (pista: entrenamiento tipo sentence-transformers)